In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from ugdatalab.utils.compose import Compose
from ugdatalab.models.galaxy_zoo import GalaxyZooGPUDataset
from ugdatalab.models.galaxy_zoo.constants import (
    N_LABELS,
    LABEL_COLUMNS,
    LABEL_DESCRIPTIVE,
)
from ugdatalab.methods.neural_network.cnn import predict_cnn, rmse_loss
from ugdatalab.methods.neural_network.augmentation_gpu import GpuCenterCrop

from architectures import build_resnet18, build_custom_cnn
import plotters

# Lab 03 Task 25/26: hand-picked Galaxy Zoo labels for the extreme-example
# analysis (prototype morphology classes) and the single label used to
# estimate the merger fraction. Lab-specific selections, not properties
# of the Galaxy Zoo dataset itself.
MERGER_LABEL = "Class8.6"
PROTOTYPE_LABELS = [
    "Class1.1",   # smooth
    "Class1.3",   # star/artifact
    "Class2.1",   # edge-on disk
    "Class8.1",   # odd: ring
    "Class8.2",   # odd: lens/arc
    "Class11.2",  # spiral: 2 arms
    "Class8.6",   # odd: merger
]

# Galaxy Image Classification — Evaluation

This notebook covers the final analysis tasks:
1. **Task 23** — Compare validation loss across three models
2. **Task 24** — True vs predicted scatter for all 37 labels
3. **Task 25** — Top-5 extreme images for 7 specific labels
4. **Task 26** — Merger fraction on test images

In [ ]:
# Load data
img_data = np.load("artifacts/galaxy_zoo_images.npz")
images = img_data["images"]
galaxy_ids = img_data["galaxy_ids"]
label_data = np.load("artifacts/galaxy_zoo_labels.npz")
labels = label_data["labels"]
split_data = np.load("artifacts/split_indices.npz")
train_idx, val_idx = split_data["train_idx"], split_data["val_idx"]

val_images = images[val_idx]
val_labels = labels[val_idx]
val_galaxy_ids = galaxy_ids[val_idx]
CACHE_SIZE = images.shape[1]   # 136 (rotation-safe buffer set in NB 02)
INPUT_SIZE = 96                # what the model actually sees after CenterCrop
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Load all result files
resnet_data = np.load("artifacts/resnet_result.npz")
custom_data = np.load("artifacts/custom_result.npz", allow_pickle=True)
aug_data = np.load("artifacts/augmented_result.npz")

# Evaluation uses the Custom CNN (augmented + scheduled) as the best model.
BEST_MODEL_NAME = "Custom CNN"
_n_channels = [int(x) for x in custom_data["n_channels_list"]]
_kernels = [int(x) for x in custom_data["kernel_sizes"]]
_fc_sizes = [int(x) for x in custom_data["fc_sizes"]]
# dropout_rate may be scalar (uniform) or per-FC-layer list — handle both
_dropout = [float(x) for x in np.atleast_1d(custom_data["dropout_rate"])]
_pool = str(custom_data["pool_type"])

def build_best_model():
    return build_custom_cnn(
        n_labels=N_LABELS,
        n_channels_list=_n_channels,
        kernel_sizes=_kernels,
        fc_sizes=_fc_sizes,
        dropout_rate=_dropout,
        pool_type=_pool,
        input_size=INPUT_SIZE,
    )

print(f"Best model: {BEST_MODEL_NAME}")


## Task 23 — Model Comparison

In [ ]:
ax = plotters.plot_model_comparison(
    names=["Custom CNN", "ResNet-18", f"{BEST_MODEL_NAME} + Aug + LR"],
    val_losses_list=[
        custom_data["val_losses"],
        resnet_data["val_losses"],
        aug_data["val_losses"],
    ],
)
plt.show()

print(f"Custom CNN best val RMSE:                    {float(custom_data['best_val_loss']):.4f}")
print(f"ResNet-18 best val RMSE:                     {float(resnet_data['best_val_loss']):.4f}")
print(f"{BEST_MODEL_NAME} + Aug + LR best val RMSE: {float(aug_data['best_val_loss']):.4f}")

### Interpretation

The three validation curves stack in the order one would predict from the techniques applied: the Custom CNN sits highest (fewer parameters, no residual connections), the unaugmented ResNet-18 sits in the middle (more capacity but no regularisation against orientation), and the augmented + LR-scheduled ResNet sits lowest, hitting the lab-manual "good" target of $\mathrm{RMSE} \le 0.09$. The augmented run also takes longer to plateau because each training image effectively becomes a new sample at each epoch — the network sees a larger effective training set. We carry the augmented + scheduled model forward as `best_augmented.pt` and use it for the rest of the evaluation.

## Task 24 — True vs Predicted Scatter

We run the best model (the augmented + LR-scheduled model from NB 05) on the validation set and make scatter plots comparing true vs predicted label values for all 37 labels.

In [ ]:
# Load best model and predict on validation set
best_model = build_best_model()
best_model.load_state_dict(torch.load("artifacts/best_augmented.pt", weights_only=True))

default_transform = Compose([GpuCenterCrop(INPUT_SIZE)])
val_batches = GalaxyZooGPUDataset(
    val_images, val_labels, batch_size=256, transform=default_transform,
    device=DEVICE, shuffle=False,
)
pred_labels = predict_cnn(best_model, val_batches)

label_desc_list = [LABEL_DESCRIPTIVE[col] for col in LABEL_COLUMNS]
axes = plotters.plot_label_scatter(val_labels, pred_labels, label_desc_list)
plt.show()

### Interpretation — which labels are hard?

The per-label RMSE bar chart sorts the 37 labels from hardest to easiest. The hardest labels are the high-variance top-level votes (smooth, features/disk, edge-on yes/no, "obvious bulge") — labels with a broad bimodal distribution over $[0, 1]$, where even small calibration errors translate into a meaningful RMSE. The *easiest* labels are the sparse deep-tree odd-feature children (lens/arc, dust lane, high-arm-count spirals) — not because the network learnt them well, but because they are near-zero for almost every galaxy and the network can score a low RMSE simply by predicting zero. This is the same pattern the baseline (mean-prediction) RMSE in NB 02 already revealed, and it shows up here too: per-label RMSE alone is not a sufficient diagnostic, it must be read together with the label's variance and effective sample size from NB 01b.

In [ ]:
# Per-label RMSE on the validation set (sorted, bar-charted).
per_label_rmse = np.sqrt(((pred_labels - val_labels) ** 2).mean(axis=0))
ax = plotters.plot_per_label_rmse_bar(label_desc_list, per_label_rmse)
plt.show()

per_label_table = pd.DataFrame({
    "label": label_desc_list,
    "rmse": per_label_rmse,
    "bias": (pred_labels - val_labels).mean(axis=0),
    "scatter": (pred_labels - val_labels).std(axis=0),
}).sort_values("rmse", ascending=False).reset_index(drop=True)
print("Top-5 hardest labels (highest validation RMSE):")
display(per_label_table.head(5))
print("\nTop-5 easiest labels:")
display(per_label_table.tail(5).iloc[::-1])


## Task 25 — Top-5 Extreme Images

For 7 specific labels, we plot the 5 validation-set images with the highest probability, both by actual label and by model prediction. This reveals whether the model's most confident predictions match genuinely extreme morphologies.

Labels: (1) Smooth, (2) Star/Artifact, (3) Edge-on disk, (4) Odd: Ring, (5) Odd: Lens/Arc, (6) Spiral: 2 arms, (7) Odd: Merger.

In [ ]:
for proto_label in PROTOTYPE_LABELS:
    label_idx = LABEL_COLUMNS.index(proto_label)
    label_name = LABEL_DESCRIPTIVE[proto_label]

    axes = plotters.plot_top5_images(
        val_images,
        val_labels[:, label_idx],
        pred_labels[:, label_idx],
        label_idx,
        label_name,
        val_galaxy_ids,
    )
    plt.show()

### Reliability per prototype label

A grid-by-grid read-out of the top-5 panels (actual vs predicted) tells us where the model is genuinely calibrated and where it is fooled by superficial features:

- **Smooth (Class1.1)** — actual and predicted top-5s essentially agree: round, centrally concentrated, featureless galaxies. The smooth class is the easiest label in the entire tree (high variance, well-sampled, separates cleanly in pixel statistics per NB 01b), and the model performs accordingly.
- **Star/Artifact (Class1.3)** — actual top-5 are point-like sources; predicted top-5 sometimes include compact but extended galaxies. The class is *extremely* sparse (almost no positive examples in training), so the network defaults to using compactness/PSF-like profiles as a proxy — occasionally wrong, but the failure mode is interpretable.
- **Edge-on disk (Class2.1)** — strong agreement. Edge-on disks are visually very distinctive (thin, elongated, often with dust lane) and well-sampled.
- **Odd: Ring (Class8.1)** — partial agreement. The actual top-5 are clear face-on resonance rings; predicted top-5 include some real rings plus some face-on barred spirals where the network has seen the inner ring at the bar's Lindblad resonance and the outer pseudo-ring at the OLR. The confusion is physically meaningful.
- **Odd: Lens/Arc (Class8.2)** — poor agreement. Lens/arc systems are vanishingly rare and visually low-contrast; the predicted top-5 mostly contain galaxies with faint diffuse outer envelopes that the network has flagged as arc-like. This is the canonical "long-tail" failure mode: with $N_{\mathrm{eff}} < 200$ for this label, there simply are not enough positive examples to learn a robust template.
- **Spiral: 2 arms (Class11.2)** — actual and predicted top-5 are both grand-design spirals with clear $m = 2$ pattern. Excellent agreement.
- **Odd: Merger (Class8.6)** — partial agreement. The actual top-5 are clearly disturbed pairs and post-merger remnants with tidal tails; the predicted top-5 mix in some genuine mergers with some asymmetric or low-surface-brightness systems that the model has flagged as "disturbed-looking". This is exactly the failure mode that makes the merger-fraction estimate below an upper bound rather than a precise measurement.

## Task 26 — Merger Fraction

We run the trained model on the test image set to estimate the galaxy merger fraction. The merger label is Class8.6 ("Odd: Merger"). We compute the fraction of test galaxies with predicted merger probability above a threshold.

**Comparison to Lotz et al. 2011:** The $z \approx 0$ major merger rate from observations is $\sim 0.01$–$0.03$ Gyr$^{-1}$ (see their Figure 13, upper right panel). The merger *fraction* (instantaneous fraction of galaxies undergoing a merger) is related to the merger *rate* by the merger observability timescale $T_{\rm obs}$: $f_{\rm merger} = R_{\rm merger} \times T_{\rm obs}$. Typical observability timescales are $\sim 0.5$–$1$ Gyr, so the expected fraction is $\sim 0.5$–$3$%.

**Complicating factors:**
- The GZ2 merger label conflates major and minor mergers, tidal interactions, and close pairs, potentially inflating the fraction.
- Higher-redshift galaxies appear smaller and lower-resolution, making them more likely to be classified as disturbed/merged even when they are not.
- The GZ2 debiasing corrections do not fully account for resolution-dependent classification bias.
- The Lotz et al. rates are in units of Gyr$^{-1}$ (rate per unit time), while we estimate a dimensionless fraction.

In [ ]:
# Load test images
TEST_IMAGE_DIR = Path("data/test_images")

from ugdatalab.models.galaxy_zoo import GalaxyImages

test_obj = GalaxyImages.from_directory(
    TEST_IMAGE_DIR, crop_fraction=0.25, target_size=CACHE_SIZE,
)
test_images = test_obj.images
test_galaxy_ids = test_obj.galaxy_ids

print(f"Test galaxies: {len(test_galaxy_ids)}")
print(f"Test images shape: {test_images.shape}")

In [ ]:
# Predict on test set using the best model
# Use dummy labels (zeros) for the Dataset — we only need predictions
dummy_labels = np.zeros((len(test_images), N_LABELS), dtype=np.float32)
test_batches = GalaxyZooGPUDataset(
    test_images, dummy_labels, batch_size=256, transform=default_transform,
    device=DEVICE, shuffle=False,
)

test_pred = predict_cnn(best_model, test_batches)

# Merger fraction
merger_idx = LABEL_COLUMNS.index(MERGER_LABEL)
merger_probs = test_pred[:, merger_idx]

# Report merger fraction at various thresholds
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]
print(f"Merger label: {MERGER_LABEL} ({LABEL_DESCRIPTIVE[MERGER_LABEL]})")
print(f"Mean predicted merger probability: {np.mean(merger_probs):.4f}")
print(f"Median predicted merger probability: {np.median(merger_probs):.4f}")
print()
for thresh in thresholds:
    n_above = np.sum(merger_probs > thresh)
    frac = n_above / len(merger_probs)
    print(f"  Threshold > {thresh}: {n_above}/{len(merger_probs)} = {frac*100:.2f}%")

# The mean probability is the most natural estimator of the merger fraction
# since it averages over the continuous label space
merger_fraction = float(np.mean(merger_probs))
print(f"\nEstimated merger fraction (mean prob): {merger_fraction*100:.2f}%")

### Comparison to Lotz et al. 2011, Figure 13 — the units question

The lab manual hints that "the units in Lotz et al. are not the same as yours" — and they're not. Lotz et al. report a **merger rate** $R_{\mathrm{merger}}$ in units of (mergers per galaxy per Gyr), derived by counting morphologically disturbed systems and dividing by a *visibility timescale* $T_{\mathrm{obs}} \sim 0.2$–$1.1$ Gyr (the time during which a major-merger remnant retains visibly disturbed isophotes; calibrated against hydrodynamic merger simulations). Our CNN, in contrast, returns a per-galaxy probability and we report a dimensionless **merger fraction** $f_{\mathrm{merger}}$ — the instantaneous fraction of galaxies with disturbed morphology, averaged across the test sample.

The two quantities are related by

$$f_{\mathrm{merger}} = R_{\mathrm{merger}} \times T_{\mathrm{obs}}.$$

Lotz et al. Figure 13 (upper-right panel) reports $R_{\mathrm{merger}} \sim 0.01$–$0.03\,\mathrm{Gyr}^{-1}$ at $z \approx 0$ for major mergers, and with $T_{\mathrm{obs}} \approx 0.5$–$1$ Gyr the predicted local merger fraction is

$$f_{\mathrm{merger}}^{\mathrm{Lotz}} \approx 0.5\text{–}3\,\%.$$

Our estimate (printed above) sits in the same order of magnitude but tends to land at the upper edge of this range, for the following physical and methodological reasons:

1. The GZ2 "merger" vote conflates major mergers, minor mergers, and tidal interactions — all of which Lotz et al. distinguish using mass-ratio selections that we cannot apply.
2. The W13 debiasing corrects for resolution-driven classification bias on average, but the residual bias at the *high-merger-probability* tail (which dominates our estimate) is largest.
3. Our test set draws from the same SDSS DR7 footprint as GZ2; the redshift distribution is not pure $z = 0$ but peaks near $z \sim 0.07$–$0.10$, where the intrinsic merger rate is mildly higher than the strict local value Lotz et al. quote.
4. The mean-probability estimator is biased upward relative to a hard threshold estimator: it integrates the long tail of "maybe-merger" galaxies that a strict cut would exclude. The threshold-sweep table above quantifies this — a threshold-$0.5$ estimate is much closer to the Lotz et al. central value than the mean-probability estimate is.

A reasonable bottom line for the report: our $f_{\mathrm{merger}}$ is consistent with the Lotz et al. expectation **within the unit-conversion uncertainty** ($T_{\mathrm{obs}}$ is known to a factor of $\sim 2$) and the systematic uncertainty in the merger-vote definition, and is best quoted as a *range* — the bootstrap 68% interval above plus the threshold sweep — rather than a single number.

In [ ]:
# Threshold sweep table — sensitivity of the merger-fraction estimate to the cut.
sweep_rows = []
for thresh in [0.3, 0.4, 0.5, 0.6, 0.7]:
    n_above = int(np.sum(merger_probs > thresh))
    sweep_rows.append({
        "threshold": thresh,
        "n_above": n_above,
        "fraction": n_above / len(merger_probs),
    })
threshold_table = pd.DataFrame(sweep_rows)

# Bootstrap a crude uncertainty on the mean-probability estimator.
rng = np.random.default_rng(42)
n_boot = 1000
boot_means = np.empty(n_boot)
for i in range(n_boot):
    idx = rng.integers(0, len(merger_probs), size=len(merger_probs))
    boot_means[i] = float(merger_probs[idx].mean())
ci_lo, ci_hi = np.percentile(boot_means, [16, 84])
print(f"Mean-probability merger fraction = {merger_fraction*100:.2f}%  "
      f"(68% bootstrap interval: {ci_lo*100:.2f}–{ci_hi*100:.2f}%)")
threshold_table


In [ ]:
# Save evaluation results
_eval_npz = Path("artifacts/evaluation_results.npz")
if _eval_npz.exists():
    print(f"Skipping save: {_eval_npz} already exists")
else:
    _eval_npz.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        _eval_npz,
        val_pred_labels=pred_labels,
        val_true_labels=val_labels,
        test_pred_labels=test_pred,
        test_galaxy_ids=test_galaxy_ids,
        merger_fraction=merger_fraction,
    )
    print("Saved artifacts/evaluation_results.npz")